# पाठ 01 - एआई एजेन्टहरूमा परिचय

**शुरुआतीहरूका लागि एआई एजेन्टहरू** कोर्सको पहिलो पाठमा स्वागत छ!

एउटा **एआई एजेन्ट** एउटा प्रोग्राम हो जुन ठूलो भाषा मोडेल (LLM) लाई यसको तर्कशक्ति इन्जिनको रूपमा प्रयोग गर्दछ र वास्तविक विश्वमा *कार्यहरू* लिन सक्छ — API कल गर्ने, डाटाबेस सोध्ने, वा कोड चलाउने — प्रयोगकर्ताको तर्फबाट लक्ष्य पुरा गर्न।

यस नोटबुकमा तपाईं आफ्नो पहिलो एजेन्ट निर्माण गर्नुहुनेछ: एउटा **ट्राभल एजेन्ट** जसले विदा गन्तव्यहरू सिफारिस गर्दछ। यस यात्रामा तपाईंले सिक्नुहुनेछ:

1. **Microsoft Agent Framework** प्रयोग गरेर Azure AI Foundry Agent Service सँग जडान गर्ने।
2. एजेन्टलाई एउटा **टुल** दिने — एक सामान्य Python फङ्सन जुन यसले कल गर्न सक्छ।
3. एजेन्ट चलाउने र यसको प्रतिक्रिया निरीक्षण गर्ने।
4. एजेन्टको प्रतिक्रियालाई टोकन-दर-टोकन स्ट्रिम गर्ने।


## सेटअप

यो नोटबुक चलाउनु भन्दा पहिले, सुनिश्चित गर्नुहोस् कि तपाईंसँग:

1. **एउटा Azure AI Foundry परियोजना** छ जुनमा एउटा सञ्‍चालित संवाद मोडेल (जस्तै `gpt-4o-mini`) तैनाथ गरिएको छ।
2. **Azure CLI सँग लगइन गरिसकिएको छ** — तपाईँको टर्मिनलमा `az login` चलाउनुहोस्।
3. **आवश्यक वातावरण चरहरू सेट गरिएको छ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तपाईँको Azure AI Foundry परियोजनाको अन्तिम बिन्दु।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईँको तैनाथ गरिएको मोडेलको नाम।

तलको सेलले तपाईँलाई आवश्यक पर्ने Python प्याकेजहरू इन्स्टल गर्दछ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## तपाईंको पहिलो एजेन्ट सिर्जना गर्दै

एजेन्टलाई दुई कुराहरू आवश्यक हुन्छ:

- **निर्देशनहरू** जुन यसलाई *को हो* र *कसरी व्यवहार गर्ने* भनेर बताउँछन् (एक सिस्टम प्रम्प्ट)।
- **उपकरणहरू** — Python फङ्सनहरू जुन `@tool` ले सजाइएको हुन्छन् जसलाई एजेन्टले जानकारी प्राप्त गर्न वा क्रियाकलापहरू गर्न कल गर्न सक्छ।

तल हामी एउटा सरल उपकरण परिभाषित गर्छौं जुन लोकप्रिय छुट्टी गन्तव्यहरूको सूची फिर्ता गर्छ। प्रयोगकर्ताले यात्रा सिफारिसहरूको लागि सोधेपछि एजेन्टले यस उपकरणलाई प्रयोग गर्नेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रिमिङ प्रतिक्रिया

अझ अन्तरक्रियात्मक अनुभवको लागि तपाईंले एजेन्टको प्रतिक्रियालाई **स्ट्रिम** गर्न सक्नु हुन्छ। पूर्ण जवाफको प्रतीक्षा गर्ने सट्टा, एजेन्टले उत्पन्न गरिएका पाठ टुक्राहरू प्रदान गर्छ। यो च्याट इन्टरफेसहरूमा विशेष गरी उपयोगी हुन्छ जहाँ तपाईंले वास्तविक समयमा आउटपुट देखाउन चाहनुहुन्छ।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

यस पाठमा तपाईंले निम्न कुरा सिक्नु भयो:

- **एउटा प्रदायक सिर्जना गर्नुहोस्** जसले `AzureAIProjectAgentProvider` मार्फत Azure AI Foundry Agent Service सँग जडान गर्दछ।
- **@tool डेकोरेटर प्रयोग गरेर एउटा उपकरण परिभाषित गर्नुहोस्** ताकि एजेन्टले तपाईंको Python कार्यहरू कल गर्न सक्छ।
- **एजेन्टलाई प्रयोगकर्ताको सन्देशसहित चलाउनुहोस्** र यसको प्रतिक्रिया प्रिन्ट गर्नुहोस्।
- **रिक्त समयमा आउटपुटका लागि प्रतिक्रियाहरू स्ट्रिम गर्नुहोस्**।

अर्को पाठमा हामी एजेन्टिक फ्रेमवर्कहरूलाई थप गहिराइमा अन्वेषण गर्नेछौं र एजेन्टहरूलाई बढी शक्तिशाली उपकरणहरू र बहु-चरण तर्क क्षमताहरू कसरी दिन सकिन्छ भनेर सिक्नेछौं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयत्न गरिरहेका छौं, तर कृपया जानकार हुनुहोस् कि स्वचालित अनुवादमा त्रुटि वा गलत अर्थ हुन सक्ने संभावना हुन्छ। मूल भाषा मा रहेको दस्तावेजलाई आधिकारिक स्रोत मान्नुपर्छ। महत्वपूर्ण जानकारीको लागि, पेशेवर मानवी अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
